Install k3s:
```bash
curl -sfL https://get.k3s.io | sh -
```

Check installation:
```bash
sudo kubectl get nodes
```

# Deploy Jitsi Meet
Jitsi Meet has 4 core components: web (frontend), prosody (XMPP server), jicofo (meeting management), and jvb (video bridge).

Create a folder for Jitsi, replace `PUBLIC_URL` with your `floating ip`:
```bash
mkdir -p ~/k8s/jitsi
```

Create environment for basic Jitsi configuration:
```bash
cat << 'EOF' > ~/k8s/jitsi/configmap.yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: jitsi-config
  namespace: jitsi
data:
  PUBLIC_URL: "http://<REPLACE_WITH_YOUR_FLOATING_IP>"
  XMPP_SERVER: "prosody"
  XMPP_DOMAIN: "meet.jitsi"
  XMPP_AUTH_DOMAIN: "auth.meet.jitsi"
  XMPP_MUC_DOMAIN: "muc.meet.jitsi"
  XMPP_INTERNAL_MUC_DOMAIN: "internal-muc.meet.jitsi"
  JVB_BREWERY_MUC: "jvbbrewery"
  JVB_PORT: "10000"
  JVB_STUN_SERVERS: "meet-jit-si-turnrelay.jitsi.net:443"
  TZ: "America/New_York"
  JICOFO_AUTH_USER: "focus"
EOF
```

Create namespace:
```bash
sudo kubectl create namespace jitsi
```

Create k8s secret, replace `<YOUR_JICOFO-PASSWORD>` and `<YOUR-JVB-PASSWORD>` with your own passwords:
```bash
sudo kubectl create secret generic jitsi-secrets \
  --from-literal=JICOFO_AUTH_PASSWORD=<YOUR-JICOFO-PASSWORD> \  
  --from-literal=JVB_AUTH_PASSWORD=<YOUR-JVB-PASSWORD> \  
  -n jitsi
```

Create prosody:
```bash
cat << 'EOF' > ~/k8s/jitsi/prosody.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: prosody
  namespace: jitsi
spec:
  replicas: 1
  selector:
    matchLabels:
      app: prosody
  template:
    metadata:
      labels:
        app: prosody
    spec:
      containers:
      - name: prosody
        image: jitsi/prosody:stable
        envFrom:
        - configMapRef:
            name: jitsi-config
        - secretRef:
            name: jitsi-secrets
        ports:
        - containerPort: 5222
        - containerPort: 5280
        - containerPort: 5347
        resources:
          requests:
            cpu: "50m"
            memory: "64Mi"
          limits:
            cpu: "200m"
            memory: "256Mi"
---
apiVersion: v1
kind: Service
metadata:
  name: prosody
  namespace: jitsi
spec:
  selector:
    app: prosody
  ports:
  - name: xmpp
    port: 5222
  - name: http
    port: 5280
  - name: component
    port: 5347
EOF
```

Create jicofo:
```bash
cat << 'EOF' > ~/k8s/jitsi/jicofo.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: jicofo
  namespace: jitsi
spec:
  replicas: 1
  selector:
    matchLabels:
      app: jicofo
  template:
    metadata:
      labels:
        app: jicofo
    spec:
      containers:
      - name: jicofo
        image: jitsi/jicofo:stable
        envFrom:
        - configMapRef:
            name: jitsi-config
        - secretRef:
            name: jitsi-secrets
        ports:
        - containerPort: 8888
        resources:
          requests:
            cpu: "50m"
            memory: "200Mi"
          limits:
            cpu: "300m"
            memory: "400Mi"
---
apiVersion: v1
kind: Service
metadata:
  name: jicofo
  namespace: jitsi
spec:
  selector:
    app: jicofo
  ports:
  - name: http
    port: 8888
EOF
```

Create jvb:
```bash
cat << 'EOF' > ~/k8s/jitsi/jvb.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: jvb
  namespace: jitsi
spec:
  replicas: 1
  selector:
    matchLabels:
      app: jvb
  template:
    metadata:
      labels:
        app: jvb
    spec:
      containers:
      - name: jvb
        image: jitsi/jvb:stable
        envFrom:
        - configMapRef:
            name: jitsi-config
        - secretRef:
            name: jitsi-secrets
        ports:
        - containerPort: 10000
          protocol: UDP
        - containerPort: 8080
        resources:
          requests:
            cpu: "50m"
            memory: "256Mi"
          limits:
            cpu: "500m"
            memory: "512Mi"
---
apiVersion: v1
kind: Service
metadata:
  name: jvb
  namespace: jitsi
spec:
  selector:
    app: jvb
  ports:
  - name: media
    port: 10000
    protocol: UDP
  - name: http
    port: 8080
EOF
```

Create web:
```bash
cat << 'EOF' > ~/k8s/jitsi/web.yaml
apiVersion: apps/v1
kind: Deployment
metadata:
  name: web
  namespace: jitsi
spec:
  replicas: 1
  selector:
    matchLabels:
      app: web
  template:
    metadata:
      labels:
        app: web
    spec:
      containers:
      - name: web
        image: jitsi/web:stable
        envFrom:
        - configMapRef:
            name: jitsi-config
        ports:
        - containerPort: 80
        - containerPort: 443
        resources:
          requests:
            cpu: "10m"
            memory: "32Mi"
          limits:
            cpu: "200m"
            memory: "128Mi"
---
apiVersion: v1
kind: Service
metadata:
  name: web
  namespace: jitsi
spec:
  type: NodePort
  selector:
    app: web
  ports:
  - name: http
    port: 80
    targetPort: 80
    nodePort: 30080
  - name: https
    port: 443
    targetPort: 443
    nodePort: 30443
EOF
```

Deploy all Jitsi Components:
```bash
sudo kubectl apply -f ~/k8s/jitsi/
```

Verify deployment:
```bash
sudo kubectl get all -n jitsi
```

Replace `A.B.C.D` with your `floating ip` and navigate to the link to acccess Jitsi Meet:  

http://A.B.C.D:30080